#### Overview and exampmles of the Nvidia Newton Simulator

As the cabalitie of physical AI systems accelrate, engineers and developers need more accurate physics simulation environemnts that reduce the sim-to-real gap. Simulation can be used to scale robot learning systems to use the collective learning of multiple robot instnaces over an accelarated period of time. Traditional CPU based physics simulators such as the original Mujuco implementation struggle to render complex environment dynamics  in simulation. Physics simulation can leverage the parallelization capabilities provided by GPUs to enable powerful and realistic physics simulation using tools like Nvidia Newton. Nvidia Newton is written using the Nvidia Warp library which allows for writing differential spatial computing kernels in python.

Newton extends and generalizes Warp's simulation moulde and intergrates Mujoco Warp as its primary backend. Newton emphasizes GPU computation, and supports the open universal scene description library OpenUSD. 

##### Building Simulation examples with Nvidia newton

To start we will look at a simple pendulum simulation provided in the netwon repository. The pendulum has two joints and swings freely in the air. 

In [ ]:

import warp as wp

import newton
import newton.examples

In [ ]:
class PendulumSimulator:
    def __init__(self, viewer):

        # setup simulation parameters first
        self.fps = 100
        self.frame_dt = 1.0 / self.fps
        self.sim_time = 0.0
        self.sim_substeps = 10
        self.sim_dt = self.frame_dt / self.sim_substeps

        # we'll store a reference to the viewer in the simulator class
        self.viewer = viewer
        
        # we'll define a model body builder object
        builder = newton.ModelBuilder()

        # add the pendulum articulation to the model
        builder.add_articulation(key="2-link-pendulum")

        # define the size of the links
        hx = 1.0
        hy = 0.1
        hz = 0.1

        # create first link
        link_0 = builder.add_body()
        builder.add_shape_box(link_0, hx=hx, hy=hy, hz=hz)
        # create second link
        link_1 = builder.add_body()
        builder.add_shape_box(link_1, hx=hx, hy=hy, hz=hz)

        # add joints
        rot = wp.quat_from_axis_angle(wp.vec3(0.0, 0.0, 1.0), -wp.pi * 0.5)
        # add the first joint
        builder.add_joint_revolute(
            parent=-1,
            child=link_0,
            axis=wp.vec3(0.0, 1.0, 0.0),
            # rotate pendulum around the z-axis to appear sideways to the viewer
            parent_xform=wp.transform(p=wp.vec3(0.0, 0.0, 5.0), q=rot),
            child_xform=wp.transform(p=wp.vec3(-hx, 0.0, 0.0), q=wp.quat_identity()),
        )
        # add the second joint
        builder.add_joint_revolute(
            parent=link_0,
            child=link_1,
            axis=wp.vec3(0.0, 1.0, 0.0),
            parent_xform=wp.transform(p=wp.vec3(hx, 0.0, 0.0), q=wp.quat_identity()),
            child_xform=wp.transform(p=wp.vec3(-hx, 0.0, 0.0), q=wp.quat_identity()),
        )

        # add ground plane
        builder.add_ground_plane()

        # finalize model
        self.model = builder.finalize()

        # implement the physics solver
        self.solver = newton.solvers.SolverXPBD(self.model)

        # initialize the state variables
        self.state_0 = self.model.state()
        self.state_1 = self.model.state()
        self.control = self.model.control()
        self.contacts = self.model.collide(self.state_0)

        self.viewer.set_model(self.model)

        # not required for MuJoCo, but required for other solvers
        newton.eval_fk(self.model, self.model.joint_q, self.model.joint_qd, self.state_0)

        self.capture()

    def capture(self):
        if wp.get_device().is_cuda:
            with wp.ScopedCapture() as capture:
                self.simulate()
            self.graph = capture.graph
        else:
            self.graph = None

    def simulate(self):
        for _ in range(self.sim_substeps):
            self.state_0.clear_forces()

            # apply forces to the model
            self.viewer.apply_forces(self.state_0)

            self.contacts = self.model.collide(self.state_0)
            self.solver.step(self.state_0, self.state_1, self.control, self.contacts, self.sim_dt)

            # swap states
            self.state_0, self.state_1 = self.state_1, self.state_0

    def step(self):
        if self.graph:
            wp.capture_launch(self.graph)
        else:
            self.simulate()
        self.sim_time += self.frame_dt

    def test(self):
        # rough check that the pendulum links are in the correct area
        newton.examples.test_body_state(
            self.model,
            self.state_0,
            "pendulum links in correct area",
            lambda q, qd: abs(q[0]) < 1e-5 and abs(q[1]) < 1.0 and q[2] < 5.0 and q[2] > 0.0,
            [0, 1],
        )

        def check_velocities(_, qd):
            # velocity outside the plane of the pendulum should be close to zero
            check = abs(qd[0]) < 1e-4 and abs(qd[6]) < 1e-4
            # velocity in the plane of the pendulum should be reasonable
            check = check and abs(qd[1]) < 10.0 and abs(qd[2]) < 5.0 and abs(qd[3]) < 10.0 and abs(qd[4]) < 10.0
            return check

        newton.examples.test_body_state(
            self.model,
            self.state_0,
            "pendulum links have reasonable velocities",
            check_velocities,
            [0, 1],
        )

    def render(self):
        self.viewer.begin_frame(self.sim_time)
        self.viewer.log_state(self.state_0)
        self.viewer.log_contacts(self.contacts, self.state_0)
        self.viewer.end_frame()


In [ ]:
if __name__ == "__main__":
    # Parse arguments and initialize viewer
    viewer, args = newton.examples.init()

    # Create viewer and run
    example = PendulumSimulator(viewer)

    newton.examples.run(example, args)

We can now observe how the model works in the simulated envirnoment using the graphical window opened blow. 
We can adjust parameters like to wind to see how changing the physical parameters of the model affects the model of the robot. 


We can now extend the basic 2 link pendulum into a 3 link pendulum and see how the simulation model changes accordingly. 

In [ ]:
class ThreeLinkPendulumSimulator:
    def __init__(self, viewer):
        # setup simulation parameters first
        self.fps = 100
        self.frame_dt = 1.0 / self.fps
        self.sim_time = 0.0
        self.sim_substeps = 10
        self.sim_dt = self.frame_dt / self.sim_substeps

        self.viewer = viewer

        builder = newton.ModelBuilder()

        builder.add_articulation(key="pendulum")

        hx = 0.7
        hy = 0.1
        hz = 0.1

        # create first link
        link_0 = builder.add_body()
        builder.add_shape_box(link_0, hx=hx, hy=hy, hz=hz)

        link_1 = builder.add_body()
        builder.add_shape_box(link_1, hx=hx, hy=hy, hz=hz)

        link_2 = builder.add_body()
        builder.add_shape_box(link_2, hx=hx, hy=hy, hz=hz)

        # add joints
        rot = wp.quat_from_axis_angle(wp.vec3(0.0, 0.0, 1.0), -wp.pi * 0.5)
        builder.add_joint_revolute(
            parent=-1,
            child=link_0,
            axis=wp.vec3(0.0, 1.0, 0.0),
            # rotate pendulum around the z-axis to appear sideways to the viewer
            parent_xform=wp.transform(p=wp.vec3(0.0, 0.0, 5.0), q=rot),
            child_xform=wp.transform(p=wp.vec3(-hx, 0.0, 0.0), q=wp.quat_identity()),
        )
        builder.add_joint_revolute(
            parent=link_0,
            child=link_1,
            axis=wp.vec3(0.0, 1.0, 0.0),
            parent_xform=wp.transform(p=wp.vec3(hx, 0.0, 0.0), q=wp.quat_identity()),
            child_xform=wp.transform(p=wp.vec3(-hx, 0.0, 0.0), q=wp.quat_identity()),
        )

        # add the third joint
        builder.add_joint_revolute(
            parent=link_1,
            child=link_2,
            axis=wp.vec3(0.0, 1.0, 0.0),
            parent_xform=wp.transform(p=wp.vec3(hx, 0.0, 0.0), q=wp.quat_identity()),
            child_xform=wp.transform(p=wp.vec3(-hx, 0.0, 0.0), q=wp.quat_identity()),
        )

        # add ground plane
        builder.add_ground_plane()

        # finalize model
        self.model = builder.finalize()

        self.solver = newton.solvers.SolverXPBD(self.model)

        self.state_0 = self.model.state()
        self.state_1 = self.model.state()
        self.control = self.model.control()
        self.contacts = self.model.collide(self.state_0)

        self.viewer.set_model(self.model)

        # not required for MuJoCo, but required for other solvers
        # add third joint
        newton.eval_fk(self.model, self.model.joint_q, self.model.joint_qd, self.state_0)

        self.capture()

    def capture(self):
        if wp.get_device().is_cuda:
            with wp.ScopedCapture() as capture:
                self.simulate()
            self.graph = capture.graph
        else:
            self.graph = None

    def simulate(self):
        for _ in range(self.sim_substeps):
            self.state_0.clear_forces()

            # apply forces to the model
            self.viewer.apply_forces(self.state_0)

            self.contacts = self.model.collide(self.state_0)
            self.solver.step(self.state_0, self.state_1, self.control, self.contacts, self.sim_dt)

            # swap states
            self.state_0, self.state_1 = self.state_1, self.state_0

    def step(self):
        if self.graph:
            wp.capture_launch(self.graph)
        else:
            self.simulate()
        self.sim_time += self.frame_dt

    def test(self):
        # rough check that the pendulum links are in the correct area
        newton.examples.test_body_state(
            self.model,
            self.state_0,
            "pendulum links in correct area",
            lambda q, qd: abs(q[0]) < 1e-5 and abs(q[1]) < 1.0 and q[2] < 5.0 and q[2] > 0.0,
            [0, 1],
        )

        def check_velocities(_, qd):
            # velocity outside the plane of the pendulum should be close to zero
            check = abs(qd[0]) < 1e-4 and abs(qd[6]) < 1e-4
            # velocity in the plane of the pendulum should be reasonable
            check = check and abs(qd[1]) < 10.0 and abs(qd[2]) < 5.0 and abs(qd[3]) < 10.0 and abs(qd[4]) < 10.0
            return check

        newton.examples.test_body_state(
            self.model,
            self.state_0,
            "pendulum links have reasonable velocities",
            check_velocities,
            [0, 1],
        )

    def render(self):
        self.viewer.begin_frame(self.sim_time)
        self.viewer.log_state(self.state_0)
        self.viewer.log_contacts(self.contacts, self.state_0)
        self.viewer.end_frame()


Nvidia newton provides a new way to model complex physical objects and mirror accurate real-world physical dynamics.